In [ ]:
import tensorflow as tf
from keras.layers import (
    Input, Dense, Embedding, LSTM, Dropout,
    RepeatVector, Concatenate, TimeDistributed
)
from keras.models import Model


def define_model(vocab_size, max_length):

    # =========================
    # IMAGE BRANCH
    # =========================
    image_input = Input(shape=(512,), name="image_features")

    image = Dropout(0.5)(image_input)

    image = Dense(256, activation="relu", name="image_projection")(image)

    h0 = Dense(256, activation="tanh", name="initial_hidden")(image)
    c0 = Dense(256, activation="tanh", name="initial_cell")(image)

    # =========================
    # CAPTION BRANCH
    # =========================
    caption_input = Input(shape=(max_length,), name="caption")

    x = Embedding(
        input_dim=vocab_size,
        output_dim=256,
        mask_zero=True,
        name="embedding"
    )(caption_input)

    x = Dropout(0.5)(x)

    x = LSTM(
        256,
        return_sequences=True,
        dropout=0.3,
        name="lstm"
    )(x, initial_state=[h0, c0])

    # =========================
    # IMAGE CONDITIONING
    # =========================
    image_sequence = RepeatVector(max_length)(image)

    x = Concatenate(axis=-1)([x, image_sequence])

    # =========================
    # CLASSIFIER
    # =========================
    x = TimeDistributed(Dense(256, activation="relu"))(x)
    x = Dropout(0.5)(x)

    outputs = TimeDistributed(Dense(vocab_size, activation="softmax"))(x)

    model = Model(
        inputs=[image_input, caption_input],
        outputs=outputs,
        name="image_caption_generator"
    )

    # =========================
    # COMPILE
    # =========================
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3), # type: ignore
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
# load config

In [ ]:
import tensorflow as tf
import numpy as np


# =========================================================
# LOAD PRECOMPUTED CNN FEATURES
# =========================================================

features = np.load("../data/images/features.npz")


# =========================================================
# TFRECORD PARSING (UNCHANGED STRUCTURE)
# =========================================================

def parse_example(proto):
    feature_desc = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "input": tf.io.FixedLenFeature([16], tf.int64),
        "target": tf.io.FixedLenFeature([16], tf.int64),
    }

    return tf.io.parse_single_example(proto, feature_desc)


# =========================================================
# SAFE FEATURE LOOKUP
# =========================================================

def load_features(example):

    image_id = example["image"]

    feature = tf.py_function(
        func=lambda x: features[x.numpy().decode()].astype(np.float32),
        inp=[image_id],
        Tout=tf.float32,
    )

    feature.set_shape((512,))  # type: ignore
    
    example["image"] = feature
    return example


# =========================================================
# PREPARE MODEL INPUT/OUTPUT
# =========================================================

def prepare(example):

    return {
        "image_features": example["image"],
        "caption": example["input"]
    }, example["target"]


# =========================================================
# DATASET PIPELINE
# =========================================================

BATCH_SIZE = 64


def build_dataset_from_tf_record(tfrecord_path, batch_size=BATCH_SIZE, dataset_size=None):
    dataset = tf.data.TFRecordDataset(tfrecord_path)

    dataset = dataset.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.map(load_features, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.map(prepare, num_parallel_calls=tf.data.AUTOTUNE)

    if dataset_size is not None:
        dataset = dataset.shuffle(dataset_size, reshuffle_each_iteration=True)

    dataset = dataset.repeat()
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

train_set = build_dataset_from_tf_record(
    "../data/records/train.tfrecord",
    dataset_size=30000, # TODO: da ricalcolare!
    batch_size=BATCH_SIZE
)

val_set = build_dataset_from_tf_record(
    "../data/records/val.tfrecord",
    batch_size=BATCH_SIZE
)



In [ ]:
# =========================
# MODEL
# =========================
model = define_model(vocab_size, max_length)


# =========================
# CALLBACKS
# =========================
early_stopping = tf.keras.callbacks.EarlyStopping( # type: ignore
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau( # type: ignore
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint( # type: ignore
    "best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

# =========================
# TRAINING
# =========================
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=[early_stopping, reduce_lr, checkpoint],
    shuffle=True
)